<a href="https://colab.research.google.com/github/kazumasaiinuma/DS340-Project/blob/main/notebooks/AIPhotoCulling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Photo Culling — Photo Triage experiments

Alex Bischof, Kazumasa Iinuma — DS340, Fall 2026

We propose an AI Photo Culling method to save time sifting through photos, which is often redundant. Photo culling is the process of reviewing all photos from a photoshoot and choosing the best shots to edit and deliver. Photo culling can take hours to days based on how many photos were taken.


**Data:** [Princeton/Adobe Photo Triage Benchmark](https://pixl.cs.princeton.edu/pubs/Chang_2016_ATF/index.php)
(Chang et al., SIGGRAPH 2016), via the Kaggle mirror `ericwolter/triage`.
Only `train_val/` is used — the official test split has no released labels.

**How to run.** Sections 1–4 need the 13.65 GB image download and a GPU, and write their
results to Google Drive. Once that has been done once, a new session can skip straight to
section 5 using the *Resume* cell below.

## 0. Resume in a new session (optional)

Colab wipes `/content` when a session ends, but `features_v1.csv`, `clip_b32.npy` and
`clip_keys.csv` were saved to Drive. They are all that sections 5 onwards need, so a new
session can skip the download and the GPU entirely.

Run this cell **instead of** sections 1–4. Expected output:
`photos 12988 | train 10170 val 483 test 1905 pairs`.
If those numbers differ, the split has drifted and the results are not comparable.

In [1]:
# Rebuilds every variable used from section 5 onward, straight from Drive.
# Skip this cell if you are running sections 1-4 in this session.
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegressionCV
from scipy.stats import binomtest

D = "/content/drive/MyDrive/ds340"
cols = ["series", "p1", "p2", "ratio", "rank1", "rank2"]
tr = pd.read_csv(f"{D}/train_pairlist.txt", sep=r"\s+", header=None, names=cols)
va = pd.read_csv(f"{D}/val_pairlist.txt",   sep=r"\s+", header=None, names=cols)

ser = pd.Series(tr.series.unique()).sample(frac=1, random_state=0)
test_s = set(ser[: int(0.15 * len(ser))])
te, tr = tr[tr.series.isin(test_s)], tr[~tr.series.isin(test_s)]

feat = pd.read_csv(f"{D}/features_v1.csv")
keys = pd.read_csv(f"{D}/clip_keys.csv")
embs = np.load(f"{D}/clip_b32.npy")
pca  = PCA(32, random_state=0).fit(embs[~keys.series.isin(test_s | set(va.series))])
clip = pd.DataFrame(pca.transform(embs), columns=[f"c{i}" for i in range(32)])
P = pd.concat([keys, clip], axis=1).merge(feat, on=["series", "photo"])
P["has_face"] = (P.n_faces > 0).astype(float)
P["blink"] = P.mp_blink_max.fillna(0)
P["smile"] = P.mp_smile_mean.fillna(0)
P = P.set_index(["series", "photo"])

FEATS = ["sharp", "blink", "smile", "has_face"] + list(clip.columns)
HAND, CLIPF = ["sharp", "blink", "smile", "has_face"], list(clip.columns)
FE = HAND + CLIPF

print(f"photos {len(P)} | train {len(tr)} val {len(va)} test {len(te)} pairs")

Mounted at /content/drive
photos 12988 | train 10170 val 483 test 1905 pairs


For if Google Drive doesn't work:

In [ ]:
from google.colab import files
# `_ =` keeps Colab from printing the return value: it holds the raw bytes of
# every uploaded file, which bloated this notebook to 88 MB.
_ = files.upload()

In [ ]:
import numpy as np, pandas as pd
from sklearn.decomposition import PCA

D = "/content"                      # files uploaded above, not Drive
cols = ["series", "p1", "p2", "ratio", "rank1", "rank2"]
tr = pd.read_csv(f"{D}/train_pairlist.txt", sep=r"\s+", header=None, names=cols)
va = pd.read_csv(f"{D}/val_pairlist.txt",   sep=r"\s+", header=None, names=cols)

ser = pd.Series(tr.series.unique()).sample(frac=1, random_state=0)
test_s = set(ser[: int(0.15 * len(ser))])
te, tr = tr[tr.series.isin(test_s)], tr[~tr.series.isin(test_s)]

feat = pd.read_csv(f"{D}/features_v1.csv")
keys = pd.read_csv(f"{D}/clip_keys.csv")
embs = np.load(f"{D}/clip_b32.npy")
pca  = PCA(32, random_state=0).fit(embs[~keys.series.isin(test_s | set(va.series))])
clip = pd.DataFrame(pca.transform(embs), columns=[f"c{i}" for i in range(32)])
P = pd.concat([keys, clip], axis=1).merge(feat, on=["series", "photo"])
P["has_face"] = (P.n_faces > 0).astype(float)
P["blink"] = P.mp_blink_max.fillna(0)
P["smile"] = P.mp_smile_mean.fillna(0)
P = P.set_index(["series", "photo"])

HAND, CLIPF = ["sharp", "blink", "smile", "has_face"], list(clip.columns)
FE = HAND + CLIPF
print(f"photos {len(P)} | train {len(tr)} val {len(va)} test {len(te)} pairs")

photos 12988 | train 10170 val 483 test 1905 pairs


## 1. Download the data

Kaggle now issues an API key string rather than a `kaggle.json` file, so the key is read
with `getpass` and kept in environment variables — it is never written into the notebook.
Re-run this cell at the start of every session.

In [ ]:
!pip install -q -U kaggle
import os, getpass
os.environ["KAGGLE_USERNAME"] = "kazumasai"   # the name in your kaggle.com/<username> URL
key = getpass.getpass("Paste Kaggle API key: ")
os.environ["KAGGLE_KEY"] = key
os.environ["KAGGLE_API_TOKEN"] = key
# A listing confirms the credentials work. ericwolter/triage need not appear in the top 5.
!kaggle datasets list -s triage | head -5

Paste Kaggle API key: ··········
ref                                                                              title                                                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------------------------------  --------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
ilkeryildiz/emergency-service-triage-application                                 Emergency Service - Triage Application                    37064  2021-08-28 13:42:23.920000           6177         73                1  
jacopoferretti/emergency-triage-venomous-bites-dataset                           Emergency Triage: Venomous Bites Dataset               14455693  2026-06-20 09:27:25.403000            334         22                1  
maalona/hospital-triage-and-patient-history-data                                 Hospital Triag

The archive is 13.65 GB. Only `train_val/` is extracted; `test_imgs/` is skipped because
the official test labels were never released, so those images cannot be scored.
Takes several minutes — if it is interrupted, delete `/content/triage.zip` and start again.

In [ ]:
!kaggle datasets download -d ericwolter/triage -p /content
!unzip -q /content/triage.zip "train_val/*" -d /content/data
!rm /content/triage.zip
!ls /content/data/train_val

Sanity check on the extracted data. 12,988 images is the full 15,545 minus the unlabelled
test series. The pair lists have one row per labelled comparison:

`series  photo1  photo2  pref_ratio(photo1 over photo2)  rank1  rank2`  — rank 1 = best in series.

In [ ]:
!ls /content/data/train_val/train_val_imgs | wc -l
!wc -l /content/data/train_val/train_pairlist.txt /content/data/train_val/val_pairlist.txt

12988
 12075 /content/data/train_val/train_pairlist.txt
   483 /content/data/train_val/val_pairlist.txt
 12558 total


Copy the small label files to Drive so later sessions can reload them without
re-downloading 13.65 GB.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ds340
!cp /content/data/train_val/*.txt /content/data/train_val/*.mat /content/drive/MyDrive/ds340/

Mounted at /content/drive


## 2. Per-photo features

One pass over all 12,988 images produces four hand-built features. None of them is trained
here — MediaPipe's FaceLandmarker is used as-is, and sharpness is a plain OpenCV filter.

| Feature | What it measures |
|---|---|
| `sharp` | `log1p` of Laplacian variance, measured **on the face box** when a face is found, otherwise on the whole image. Measuring the whole frame would penalise a sharp portrait with a deliberately blurred background. |
| `mp_blink_max` | Worst eye-closure score across all faces, from MediaPipe blendshapes. Continuous, so half-closed eyes count, not only fully shut ones. |
| `mp_smile_mean` | Mean smile score across faces. |
| `n_faces` | How many faces were detected (becomes `has_face`). |

Blink and smile come free: the FaceLandmarker call was needed anyway to locate the face box
for sharpness, and blendshapes are returned by the same call.

**Note on `sharp`:** Laplacian variance responds to scene *content* as well as focus — a
brick wall scores higher than a clear sky at identical focus. That makes absolute thresholds
across a library unreliable, but within a burst the content is nearly constant, so the
confound largely cancels. This is an argument for comparing photos within a group rather
than scoring them independently.

Takes about 8 minutes on CPU. The result is cached to Drive.

In [ ]:
!pip install -q -U mediapipe opencv-python-headless
!wget -q -O /content/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

import cv2, glob, os, numpy as np, pandas as pd, mediapipe as mp
from mediapipe.tasks import python as mpt
from mediapipe.tasks.python import vision
from tqdm import tqdm

IMG = "/content/data/train_val/train_val_imgs"
lm = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mpt.BaseOptions(model_asset_path="/content/face_landmarker.task"),
    output_face_blendshapes=True, num_faces=6, min_face_detection_confidence=0.5))

def lap_var(g):
    # Laplacian = edge detector. Sharp photo -> wide spread of edge values -> high variance.
    return cv2.Laplacian(g, cv2.CV_64F).var()

rows = []
for path in tqdm(sorted(glob.glob(f"{IMG}/*.JPG"))):
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    res = lm.detect(mp.Image(image_format=mp.ImageFormat.SRGB,
                             data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
    face_sharp, blink, smile = [], [], []
    for pts, bs in zip(res.face_landmarks, res.face_blendshapes):
        # Bounding box around this face's landmarks, clipped to the image.
        xs = [p.x for p in pts]
        ys = [p.y for p in pts]
        x0, x1 = max(0, int(min(xs) * w)), min(w, int(max(xs) * w))
        y0, y1 = max(0, int(min(ys) * h)), min(h, int(max(ys) * h))
        crop = gray[y0:y1, x0:x1]
        if crop.size > 400:          # ignore faces too small to measure sharpness on
            face_sharp.append(lap_var(crop))
        b = {c.category_name: c.score for c in bs}
        blink.append(max(b["eyeBlinkLeft"], b["eyeBlinkRight"]))   # worst eye on this face
        smile.append((b["mouthSmileLeft"] + b["mouthSmileRight"]) / 2)
    s, p = os.path.basename(path)[:-4].split("-")   # filename is SERIES-PHOTO.JPG
    rows.append(dict(series=int(s), photo=int(p), n_faces=len(blink),
                     face_sharp=max(face_sharp) if face_sharp else np.nan,  # sharpest face
                     global_sharp=lap_var(gray),
                     mp_blink_max=max(blink) if blink else np.nan,   # is anyone blinking?
                     mp_smile_mean=np.mean(smile) if smile else np.nan))

feat = pd.DataFrame(rows)
# Face sharpness where a face exists, whole-image sharpness otherwise.
# log1p because raw Laplacian variance spans orders of magnitude.
feat["sharp"] = np.log1p(feat.face_sharp.fillna(feat.global_sharp))
feat.to_csv("/content/drive/MyDrive/ds340/features_v1.csv", index=False)
feat.describe()

100%|██████████| 12988/12988 [07:48<00:00, 27.73it/s]


             series         photo       n_faces   face_sharp  global_sharp  \count  12988.000000  12988.000000  12988.000000  1788.000000  12988.000000   mean    2844.751694      1.985371      0.191022   470.573478   1591.500851   std     1724.944417      1.076329      0.539719   508.911005   2079.071691   min        1.000000      1.000000      0.000000     9.612083      1.353272   25%     1349.000000      1.000000      0.000000   151.080126    329.388474   50%     2754.500000      2.000000      0.000000   318.257944    828.604371   75%     4338.000000      2.000000      0.000000   598.305679   2010.081900   max     5953.000000      8.000000      5.000000  4906.638152  23985.954208          mp_blink_max  mp_smile_mean         sharp  count   1788.000000   1.788000e+03  12988.000000  mean       0.268723   4.554925e-01      6.617813  std        0.196973   3.417500e-01      1.301912  min        0.001731   9.663835e-08      0.855807  25%        0.127922   5.221378e-02      5.755465  50%    

## 3. Baseline — "keep the sharpest photo"

The simplest possible rule, and the bar everything else has to clear: in each pair, predict
that the sharper photo wins. No training, no learned weights.

"Confident" pairs are those where at least 70% of annotators agreed. Chang et al. report
their headline numbers on that subset, so it is reported throughout.

This cell also prints the single most consequential number in the project: **only 13.8% of
images contain a detected face**, which caps how much the face-derived features can
possibly matter on this dataset.

In [ ]:
cols = ["series", "p1", "p2", "ratio", "rank1", "rank2"]
val = pd.read_csv("/content/data/train_val/val_pairlist.txt", sep=r"\s+", header=None, names=cols)
s = feat.set_index(["series", "photo"]).sharp
val["pred"] = [s[(r.series, r.p1)] > s[(r.series, r.p2)] for r in val.itertuples()]
val["truth"] = val.ratio > 0.5                     # did most people prefer photo 1?
conf = (val.ratio >= 0.7) | (val.ratio <= 0.3)     # pairs with clear human agreement
print(f"All pairs:       {(val.pred == val.truth).mean():.3f}  (n={len(val)})")
print(f"Confident pairs: {(val.pred == val.truth)[conf].mean():.3f}  (n={conf.sum()})")
print(f"Images with a face: {(feat.n_faces > 0).mean():.1%}")

All pairs:       0.553  (n=483)
Confident pairs: 0.592  (n=292)
Images with a face: 13.8%


## 4. CLIP embeddings

CLIP ViT-B/32 is a pretrained image encoder. Each photo becomes a 512-number vector
describing its contents and appearance. The encoder is **frozen** — nothing is trained here.
These vectors are what let the ranker see beyond sharpness.

Needs a GPU runtime (*Runtime → Change runtime type → T4 GPU*). About 7 minutes.
Results are cached to Drive, so this only has to run once.

In [ ]:
!pip install -q open_clip_torch
import open_clip, torch, numpy as np, pandas as pd, glob, os
from PIL import Image
from tqdm import tqdm
from google.colab import drive
drive.mount('/content/drive')

dev = "cuda"
model, _, prep = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
model = model.to(dev).eval()

paths = sorted(glob.glob("/content/data/train_val/train_val_imgs/*.JPG"))
embs = []
with torch.no_grad():                               # inference only, no gradients
    for i in tqdm(range(0, len(paths), 128)):
        batch = torch.stack([prep(Image.open(p).convert("RGB")) for p in paths[i:i + 128]]).to(dev)
        e = model.encode_image(batch)
        embs.append((e / e.norm(dim=-1, keepdim=True)).cpu().numpy())   # unit-normalise

embs = np.concatenate(embs)
# keys[i] records which (series, photo) row i of embs belongs to
keys = pd.DataFrame([os.path.basename(p)[:-4].split("-") for p in paths],
                    columns=["series", "photo"]).astype(int)
np.save("/content/drive/MyDrive/ds340/clip_b32.npy", embs)
keys.to_csv("/content/drive/MyDrive/ds340/clip_keys.csv", index=False)

Mounted at /content/drive


100%|██████████| 102/102 [07:17<00:00,  4.29s/it]


## 5. The pairwise ranker

The core model. For each labelled pair it computes `features(photo1) - features(photo2)` and
fits logistic regression to predict which photo people preferred. Working on *differences* is
what makes this a ranker rather than a quality scorer: it only ever compares two shots of the
same scene.

Design decisions in this cell:

- **Split by series, never by pair.** 15% of training *series* are held out as our own test
  set. Splitting by pair would put the same photos in both train and test.
- **PCA to 32 dimensions**, fitted on training photos only. 512 raw CLIP dimensions would
  overfit 12,075 pairs.
- **Sample weights `|ratio - 0.5|`** so near-coin-flip pairs barely influence the fit.
- **Symmetric augmentation**: every pair is also added swapped, with the label flipped, so
  the model cannot learn a bias toward whichever photo is listed first.

This is the Bradley-Terry / RankNet formulation: because the model is linear on differences,
each photo gets a single scalar score `w · x`, and the model prefers whichever score is higher.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegressionCV
D = "/content/drive/MyDrive/ds340"
cols = ["series", "p1", "p2", "ratio", "rank1", "rank2"]
tr = pd.read_csv(f"{D}/train_pairlist.txt", sep=r"\s+", header=None, names=cols)
va = pd.read_csv(f"{D}/val_pairlist.txt", sep=r"\s+", header=None, names=cols)

# Hold out 15% of training SERIES as our own test set (official test labels are unreleased)
ser = pd.Series(tr.series.unique()).sample(frac=1, random_state=0)
test_s = set(ser[: int(0.15 * len(ser))])
te, tr = tr[tr.series.isin(test_s)], tr[~tr.series.isin(test_s)]

# Per-photo feature table: hand features + CLIP reduced to 32 dims
feat = pd.read_csv(f"{D}/features_v1.csv")
keys = pd.read_csv(f"{D}/clip_keys.csv")
embs = np.load(f"{D}/clip_b32.npy")
tr_mask = ~keys.series.isin(test_s | set(va.series))      # fit PCA on training photos only
pca = PCA(32, random_state=0).fit(embs[tr_mask])
clip = pd.DataFrame(pca.transform(embs), columns=[f"c{i}" for i in range(32)])
P = pd.concat([keys, clip], axis=1).merge(feat, on=["series", "photo"])
P["has_face"] = (P.n_faces > 0).astype(float)
P["blink"] = P.mp_blink_max.fillna(0)     # no face -> no blinking to penalise
P["smile"] = P.mp_smile_mean.fillna(0)
P = P.set_index(["series", "photo"])
FEATS = ["sharp", "blink", "smile", "has_face"] + list(clip.columns)

def pairs(df, symmetric=False):
    X = P.loc[list(zip(df.series, df.p1)), FEATS].values - P.loc[list(zip(df.series, df.p2)), FEATS].values
    y = (df.ratio.values > 0.5).astype(int)
    w = np.abs(df.ratio.values - 0.5)     # ambiguous pairs count for little
    if symmetric:  # add each pair swapped, so order can't matter
        X, y, w = np.vstack([X, -X]), np.concatenate([y, 1 - y]), np.concatenate([w, w])
    return X, y, w

Xtr, ytr, wtr = pairs(tr, symmetric=True)
clf = LogisticRegressionCV(Cs=10, max_iter=2000).fit(Xtr, ytr, sample_weight=wtr)

si = FEATS.index("sharp")                 # column holding the sharpness difference
for name, df in [("val", va), ("test", te)]:
    X, y, _ = pairs(df)
    conf = ((df.ratio >= 0.7) | (df.ratio <= 0.3)).values
    pred = clf.predict(X)
    sharp_pred = (X[:, si] > 0).astype(int)        # baseline: sharper photo wins
    print(f"{name:5s} ranker all={np.mean(pred == y):.3f} conf={np.mean((pred == y)[conf]):.3f} | "
          f"sharpest all={np.mean(sharp_pred == y):.3f} conf={np.mean((sharp_pred == y)[conf]):.3f}  n={len(df)}")

val   ranker all=0.631 conf=0.743 | sharpest all=0.553 conf=0.592  n=483
test  ranker all=0.652 conf=0.726 | sharpest all=0.597 conf=0.637  n=1905


**Result.** On test, 0.652 on all pairs and 0.726 on confident pairs, against 0.597 / 0.637
for the sharpness rule. For reference, Chang et al. report 0.57 for the best prior baseline
and 0.73 for their own fine-tuned Siamese VGG. Our confident-pair figure is *comparable to*
theirs — with roughly 1,150 confident test pairs the standard error is about ±1.3 points, so
the two are statistically indistinguishable.

## 6. Ablations — which features actually matter?

Three models, same data: hand features alone, CLIP alone, and both. This is what decides
whether the feature engineering was worth doing.

In [2]:
HAND = ["sharp", "blink", "smile", "has_face"]    # features we specified by hand
CLIPF = list(clip.columns)                        # 32 learned, uninterpretable dimensions

def pairs_f(df, feats, symmetric=False):
    # Same as pairs(), but the feature list is an argument so we can ablate.
    X = (P.loc[list(zip(df.series, df.p1)), feats].values
         - P.loc[list(zip(df.series, df.p2)), feats].values)
    y = (df.ratio.values > 0.5).astype(int)
    w = np.abs(df.ratio.values - 0.5)
    if symmetric:
        X, y, w = np.vstack([X, -X]), np.concatenate([y, 1 - y]), np.concatenate([w, w])
    return X, y, w

for name, feats in [("hand only", HAND), ("CLIP only", CLIPF), ("both", HAND + CLIPF)]:
    Xtr, ytr, wtr = pairs_f(tr, feats, symmetric=True)
    m = LogisticRegressionCV(Cs=10, max_iter=2000).fit(Xtr, ytr, sample_weight=wtr)
    out = []
    for split, df in [("val", va), ("test", te)]:
        X, y, _ = pairs_f(df, feats)
        conf = ((df.ratio >= 0.7) | (df.ratio <= 0.3)).values
        pred = m.predict(X)
        out.append(f"{split} all={np.mean(pred==y):.3f} conf={np.mean((pred==y)[conf]):.3f}")
    print(f"{name:10s} " + " | ".join(out))

hand only  val all=0.576 conf=0.623 | test all=0.599 conf=0.646
CLIP only  val all=0.609 conf=0.702 | test all=0.641 conf=0.718
both       val all=0.631 conf=0.743 | test all=0.652 conf=0.726


Note that **hand-only (0.599) is barely above the single sharpest-photo rule (0.597)** —
learning weights over four features adds almost nothing, which suggests sharpness is doing
all the work within that block.

Next: is the +1.1 point gain of "both" over "CLIP only" real? Both models are scored on the
*same* pairs, so the right test is **McNemar's**, which counts only the pairs where the two
models disagree — not an independent-sample standard error.

The second half restricts the comparison to pairs where **both** photos contain a face. That
is the only place the eye and expression features can possibly help, and it was decided
before looking at the result, so it is a pre-specified subgroup rather than a fishing trip.

In [3]:
from scipy.stats import binomtest

def fit(feats):
    Xtr, ytr, wtr = pairs_f(tr, feats, symmetric=True)
    return LogisticRegressionCV(Cs=10, max_iter=2000).fit(Xtr, ytr, sample_weight=wtr)

def mcnemar(base, new, label):
    # Counts only the pairs where the two models disagree.
    b = int((~base & new).sum()); c = int((base & ~new).sum())
    p = binomtest(b, b + c, 0.5).pvalue if b + c else float("nan")
    print(f"{label}: hand features fix {b}, break {c}, p = {p:.4f}")

m_clip, m_both = fit(CLIPF), fit(HAND + CLIPF)
X1, y, _ = pairs_f(te, CLIPF)
X2, _, _ = pairs_f(te, HAND + CLIPF)
c1, c2 = m_clip.predict(X1) == y, m_both.predict(X2) == y   # per-pair correctness

mcnemar(c1, c2, "all test pairs")

has = P.has_face
mask = np.array([has[(r.series, r.p1)] > 0 and has[(r.series, r.p2)] > 0
                 for r in te.itertuples()])                 # pairs where BOTH photos have a face
print(f"\nboth-face pairs: {mask.sum()} of {len(te)} test pairs")
if mask.sum():
    print(f"  CLIP only {c1[mask].mean():.3f} | both {c2[mask].mean():.3f}")
    mcnemar(c1[mask], c2[mask], "  face pairs")

all test pairs: hand features fix 125, break 103, p = 0.1642

both-face pairs: 255 of 1905 test pairs
  CLIP only 0.635 | both 0.686
  face pairs: hand features fix 25, break 12, p = 0.0470


**p = 0.164 overall — not significant.** On this dataset the learned embeddings largely
subsume the hand-built features. On face pairs, +5.1 points at p = 0.047 — real, but fragile:
only 37 pairs are discordant, so three flips the other way would push p above 0.15.

The gain on face pairs could still be entirely sharpness, because `sharp` switches from
whole-image to face-crop measurement on exactly those pairs. The next cell separates them by
adding one feature at a time.

In [ ]:
# Add each hand feature to CLIP on its own, to see which one drives the face-pair gain.
for name, extra in [("+sharp", ["sharp"]), ("+blink", ["blink"]),
                    ("+smile", ["smile"]), ("+all hand", HAND)]:
    m = fit(CLIPF + extra)
    X, y, _ = pairs_f(te, CLIPF + extra)
    ok = m.predict(X) == y
    print(f"{name:10s} all={ok.mean():.3f}  face={ok[mask].mean():.3f}  (CLIP only: {c1.mean():.3f} / {c1[mask].mean():.3f})")

+sharp     all=0.650  face=0.671  (CLIP only: 0.641 / 0.635)
+blink     all=0.644  face=0.667  (CLIP only: 0.641 / 0.635)
+smile     all=0.639  face=0.639  (CLIP only: 0.641 / 0.635)
+all hand  all=0.652  face=0.686  (CLIP only: 0.641 / 0.635)


Sharpness (+3.6) and blink (+3.2) are the two that move the face-pair number; together with
the rest they give +5.1, so they overlap partly but neither is a proxy for the other. Section
8 shows that sharpness alone does not reach significance (p = 0.136), so the gain is a
combined effect, not attributable to any single feature. Expression contributes nothing
(+0.4 on face pairs, negative overall).

**Consequence for the project plan.** MediaPipe's blink score is the untrained version of
what a CEW-trained CNN would produce, so it is the *baseline* that model has to beat — and
the measured headroom is small: +3.2 points on 13.4% of pairs is about +0.4 points overall.
FER-2013 is not worth training at all: it is 48×48 grayscale, so face crops would have to be
downscaled heavily to use it, competing against full-resolution blendshapes that already add
nothing.

## 7. Top-1 accuracy — the metric the product actually needs

Pairwise accuracy answers "given two photos, which wins". A culling tool answers "which one
do I keep out of five". Those are different, and the `rank1` / `rank2` columns in the pair
lists give the ground truth for the second: rank 1 is the majority's favourite in each series.

Because the model is linear on differences, each photo has a single score `w · x`, and the
group winner is simply the highest score. Running a pairwise tournament instead would give
identical results — a scalar score induces a total order, so no cycles are possible.

In [ ]:
w = m_both.coef_[0]
FE = HAND + CLIPF
P["score"] = P[FE].values @ w          # per-photo score; only meaningful WITHIN a series

# Recover each photo's rank from the pair list (rank 1 = series favourite)
rk = {}
for r in te.itertuples():
    rk[(r.series, r.p1)] = r.rank1
    rk[(r.series, r.p2)] = r.rank2

by_series = {}
for (s, p), rank in rk.items():
    by_series.setdefault(s, []).append((p, rank))

hits, sharp_hits, sizes = [], [], []
for s, items in by_series.items():
    truth = [p for p, rank in items if rank == 1]
    if not truth:
        continue
    photos = [p for p, _ in items]
    sizes.append(len(photos))
    hits.append(max(photos, key=lambda p: P.score[(s, p)]) == truth[0])
    sharp_hits.append(max(photos, key=lambda p: P.sharp[(s, p)]) == truth[0])

print(f"series: {len(hits)}, mean size {np.mean(sizes):.1f}, chance {1/np.mean(sizes):.3f}")
print(f"top-1  ranker {np.mean(hits):.3f} | sharpest {np.mean(sharp_hits):.3f}")

series: 684, mean size 2.7, chance 0.373
top-1  ranker 0.575 | sharpest 0.520


0.575 against a chance rate of 0.373. Picking 1-of-n is harder than 1-of-2 because the true
favourite has to beat *every* other shot.

How much of the remaining error is the model's fault, and how much is an ambiguous target?
The next cell restricts to series where the favourite beat every rival by at least 70%.

In [ ]:
k2, conf_hits = [], []
for s, items in by_series.items():
    truth = [p for p, rank in items if rank == 1]
    if not truth:
        continue
    best = truth[0]
    ranked = sorted([p for p, _ in items], key=lambda p: -P.score[(s, p)])
    k2.append(best in ranked[:2])                 # does the favourite survive a "keep 2" cut?
    rows = te[te.series == s]
    rel = rows[(rows.p1 == best) | (rows.p2 == best)]
    margin = [r.ratio if r.p1 == best else 1 - r.ratio for r in rel.itertuples()]
    if margin and min(margin) >= 0.7:             # humans agreed decisively on this series
        conf_hits.append(ranked[0] == best)

print(f"top-1, confident series only: {np.mean(conf_hits):.3f}  (n={len(conf_hits)})")
print(f"favorite retained in top 2:   {np.mean(k2):.3f}")

top-1, confident series only: 0.666  (n=377)
favorite retained in top 2:   0.908


0.666 on decisive series versus 0.575 overall: roughly 45% of series have no clear human
favourite, so the model is scored against a coin flip on much of the set.

The 0.908 retention figure is **inflated**. A series of two photos cannot lose its favourite
to a "keep top 2" cut — nothing is discarded — so those series score 1.0 for free. With a
mean series size of 2.7, many are exactly that case. Restricting to series of three or more
is the only honest version.

In [ ]:
k2_big = []
for s, items in by_series.items():
    truth = [p for p, rank in items if rank == 1]
    photos = [p for p, _ in items]
    if not truth or len(photos) < 3:   # 2-photo series retain the favourite by construction
        continue
    ranked = sorted(photos, key=lambda p: -P.score[(s, p)])
    k2_big.append(truth[0] in ranked[:2])
print(f"series with >=3 photos: {len(k2_big)}, favorite in top 2: {np.mean(k2_big):.3f}")

series with >=3 photos: 283, favorite in top 2: 0.777


**0.777 — the favourite is lost in 22% of real multi-shot groups.** This is the number that
decides the product design, and it rules out deleting anything automatically.

The last cell maps the whole trade-off: how many photos can be discarded at each level of
retention, comparing a fixed "keep top k" rule against a relative cut-off that keeps every
photo within δ of the group's best score.

In [ ]:
rows = []
for s, items in by_series.items():
    truth = [p for p, rank in items if rank == 1]
    photos = [p for p, _ in items]
    if not truth or len(photos) < 3:
        continue
    sc = {p: float(P.score[(s, p)]) for p in photos}
    rows.append((truth[0], sc, max(sc.values()), len(photos)))

gaps = np.array([mx - v for _, sc, mx, _ in rows for v in sc.values()])
print("rule      kept%  retention")
for d in np.quantile(gaps[gaps > 0], [0.1, 0.25, 0.5, 0.75, 0.9]):
    kept = tot = hit = 0
    for best, sc, mx, n in rows:
        keep = [p for p, v in sc.items() if v >= mx - d]     # within delta of the group best
        kept += len(keep); tot += n; hit += best in keep
    print(f"d={d:6.3f}  {kept/tot:.3f}  {hit/len(rows):.3f}")
for k in [1, 2, 3]:
    kept = tot = hit = 0
    for best, sc, mx, n in rows:
        keep = sorted(sc, key=lambda p: -sc[p])[:k]
        kept += len(keep); tot += n; hit += best in keep
    print(f"top-{k}     {kept/tot:.3f}  {hit/len(rows):.3f}")

rule      kept%  retention
d= 0.194  0.348  0.551
d= 0.433  0.457  0.671
d= 0.919  0.638  0.827
d= 1.521  0.819  0.951
d= 2.497  0.927  0.982
top-1     0.274  0.435
top-2     0.549  0.777
top-3     0.823  0.933


Compared at matched discard rates, the δ rule is no better: 0.750 vs 0.777 at ~55% kept,
0.951 vs 0.933 at ~82% kept. That is expected in hindsight — δ is a monotone transform of the
model's predicted win probability, so thresholding on it partitions each group identically.
**Fixed top-k is kept for simplicity.**

## 8. Sharpness on its own

Sharpness is the feature this half of the project is responsible for, so it is also reported
without the other three. Requires sections 5–6 to have run (uses `fit`, `pairs_f`, `mask`).

In [ ]:
for name, feats in [("CLIP only", CLIPF), ("CLIP + sharp", CLIPF + ["sharp"])]:
    m = fit(feats)
    X2, y2, _ = pairs_f(te, feats)
    ok = m.predict(X2) == y2
    cf = ((te.ratio >= 0.7) | (te.ratio <= 0.3)).values
    print(f"{name:13s} test all={ok.mean():.3f} conf={ok[cf].mean():.3f} face={ok[mask].mean():.3f}")

# Is the sharpness contribution significant on its own?
ok_sharp = fit(CLIPF + ["sharp"]).predict(pairs_f(te, CLIPF + ["sharp"])[0]) == y
mcnemar(c1, ok_sharp, "CLIP vs CLIP+sharp, all pairs")
mcnemar(c1[mask], ok_sharp[mask], "CLIP vs CLIP+sharp, face pairs")

CLIP only     test all=0.641 conf=0.718 face=0.635
CLIP + sharp  test all=0.650 conf=0.721 face=0.671
CLIP vs CLIP+sharp, all pairs: hand features fix 110, break 92, p = 0.2316
CLIP vs CLIP+sharp, face pairs: hand features fix 19, break 10, p = 0.1360


### Is sharpness redundant with CLIP?

Section 8 found no significant gain from adding sharpness to CLIP, yet sharpness beats chance
as a standalone rule. One explanation is that the CLIP embedding already encodes it. Ridge
regression from the 32 CLIP dimensions to each hand feature, scored on held-out photos, tests
that directly.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

ALPHAS = np.logspace(-3, 3, 13)
tr_ph = ~P.index.get_level_values("series").isin(test_s | set(va.series))
te_ph = P.index.get_level_values("series").isin(test_s)

print("Predicting each hand feature from the 32 CLIP dimensions (held-out photos):")
for f in HAND:
    r = RidgeCV(alphas=ALPHAS).fit(P.loc[tr_ph, CLIPF].values, P.loc[tr_ph, f].values)
    r2 = r2_score(P.loc[te_ph, f].values, r.predict(P.loc[te_ph, CLIPF].values))
    print(f"  {f:9s} R^2 = {r2:.3f}")

has = (P.has_face > 0).values
print("\nFace photos only:")
for f in ["blink", "smile"]:
    a, b = tr_ph & has, te_ph & has
    r = RidgeCV(alphas=ALPHAS).fit(P.loc[a, CLIPF].values, P.loc[a, f].values)
    r2 = r2_score(P.loc[b, f].values, r.predict(P.loc[b, CLIPF].values))
    print(f"  {f:9s} R^2 = {r2:.3f}  (n_train={a.sum()}, n_test={b.sum()})")

Predicting each hand feature from the 32 CLIP dimensions (held-out photos):
  sharp     R^2 = 0.420
  blink     R^2 = 0.359
  smile     R^2 = 0.423
  has_face  R^2 = 0.591

Face photos only:
  blink     R^2 = 0.080  (n_train=1429, n_test=286)
  smile     R^2 = 0.359  (n_train=1429, n_test=286)


## 9. Explaining a score to the user

Because the model is linear, a photo's score decomposes into one term per feature:
`weight x feature_difference`. That is what makes it possible to tell a user *why* a photo
lost — something an MLP or a fine-tuned CNN could not do as directly.

Two caveats:

- The model is trained on differences, so the absolute score has no meaningful zero. Scores
  are comparable **only within a group**.
- The 32 CLIP dimensions are individually meaningless, so they are summed into one
  "appearance" term.

The sigmoid of the total margin is the model's own output: the probability that the user
would prefer this photo over the group winner.

In [ ]:
NAMED = ["sharp", "blink", "smile", "has_face"]
w = dict(zip(FE, m_both.coef_[0]))

def explain(series, cand, winner):
    # Why did `cand` lose to `winner` in this series? One term per feature.
    d = lambda f: P.loc[(series, cand), f] - P.loc[(series, winner), f]
    parts = {f: w[f] * d(f) for f in NAMED}                 # named contributions
    parts["appearance (CLIP)"] = sum(w[f] * d(f) for f in CLIPF)   # 32 dims lumped
    parts["P(prefer over top pick)"] = 1 / (1 + np.exp(-sum(parts.values())))
    return parts

# Demo on the first test series with 3+ photos (needs section 7 for P["score"]).
demo = next(s for s, items in by_series.items() if len(items) >= 3)
order = sorted([p for p, _ in by_series[demo]], key=lambda p: -P.score[(demo, p)])
print(f"series {demo}: photo {order[1]} lost to photo {order[0]}")
for k, v in explain(demo, order[1], order[0]).items():
    print(f"  {k:26s} {v:+.3f}")

series 2: photo 2 lost to photo 3
  sharp                      +0.096
  blink                      -0.000
  smile                      +0.000
  has_face                   +0.000
  appearance (CLIP)          -0.430
  P(prefer over top pick)    +0.417


## 10. Feature standardisation

The features sit on very different scales: `sharp` has std 1.30, `blink` about 0.12, and the
CLIP PCA components less still, since the embeddings were unit-normalised before PCA.
`LogisticRegressionCV` applies L2 regularisation, which penalises the sum of squared weights,
so a small-scale feature needs a large weight for a given effect and is shrunk hardest. The
penalty therefore falls unevenly for reasons that have nothing to do with how informative a
feature is — worth checking, because the headline claim here is a *null* result.

The result below was produced with `RUN_STANDARDISATION_CHECK = True`; the flag is reset to
`False` afterwards because the cell divides `P` in place and running it twice would corrupt
the feature table. Nothing moves by more than 0.4 points, so all results above use
unstandardised features.

In [5]:
RUN_STANDARDISATION_CHECK = False    # set True only after re-running section 6

if RUN_STANDARDISATION_CHECK:
    tr_photos = ~P.index.get_level_values("series").isin(test_s | set(va.series))
    P[FE] = P[FE] / P.loc[tr_photos, FE].std()      # scale by TRAINING std only
    for name, feats in [("CLIP only", CLIPF), ("both", HAND + CLIPF)]:
        m = fit(feats)
        X, y_, _ = pairs_f(te, feats)
        ok = m.predict(X) == y_
        print(f"{name:10s} all={ok.mean():.3f} face={ok[mask].mean():.3f}")

CLIP only  all=0.637 face=0.639
both       all=0.655 face=0.686


## Conclusions — Photo Triage experiments

**Setup.** Princeton–Adobe Photo Triage benchmark, `train_val` only (the official
test split has no released labels). 12,988 images; 12,075 training pairs, 483
validation pairs, and 1,905 test pairs held out as 15% of *series* (never split
within a series). The test split covers 684 series, mean size 2.7 photos.
"Confident" = pairs where ≥70% of annotators agreed.

**1. A pairwise ranker over frozen CLIP features clearly beats a sharpness rule.**
Test pairwise accuracy rises from 0.597 (pick the sharpest photo) to 0.652, and
from 0.637 to 0.726 on confident pairs. The confident-pair figure is comparable to
the 0.73 reported by Chang et al. for a fine-tuned Siamese VGG, and well above the
0.57 they report for the best prior baseline.

**2. Hand-crafted quality features do not improve the ranker overall.** CLIP alone
reaches 0.641 / 0.718 (all / confident); adding sharpness, blink, smile and
has_face reaches 0.652 / 0.726. McNemar's test on the same test pairs gives 125
corrections against 103 regressions, p = 0.164 — not significant. On this dataset,
learned embeddings largely subsume the hand-built features.

**3. The features contribute only on photos containing faces, and only in combination.**
Only 13.8% of images have a detected face, and just 255 of 1,905 test pairs have faces in
both photos. On that subset accuracy rises from 0.635 to 0.686 (McNemar 25 / 12, p = 0.047).
Added individually neither reaches significance: sharpness gives 0.671 (p = 0.136) and blink
0.667, while expression gives 0.639 — no better than CLIP alone.

**4. Sharpness is partly redundant with CLIP; eye state is not.** Ridge regression from the
32 CLIP dimensions predicts log-sharpness at R² = 0.42 on held-out photos, so the embedding
already carries much of that signal. The same regression predicts eye closure at R² = 0.08 on
photos containing a face. Eye state therefore carries information the embedding lacks, and its
small effect on the ranker reflects face coverage rather than redundancy. (The R² = 0.36 for
blink across all photos is an artifact of filling blink with 0 where no face is found; that
regression is mostly detecting faces, which CLIP does at R² = 0.59.)

**5. Selecting the single best photo is substantially harder than pairwise
comparison.** Top-1 accuracy over 684 test series is 0.575 (chance 0.373; sharpest
photo 0.520), rising to 0.666 on the 377 series where annotators agreed decisively.
The gap indicates roughly 45% of series have no clear human favourite, so the model
is being scored against an ambiguous target on much of the set.

**6. Retention degrades sharply once groups contain more than two photos.** Across
all series, the favourite survives a "keep top 2" cut 90.8% of the time, but that
figure is inflated by two-photo series, where no photo is discarded. Restricted to
the 283 series with three or more photos:

| Rule | Photos kept | Favourite retained |
|---|---|---|
| Keep 1 | 27% | 0.435 |
| Keep 2 | 55% | 0.777 |
| Keep 3 | 82% | 0.933 |

Holding 95% retention permits discarding only about 18% of photos. We conclude the
model is not accurate enough for destructive automated culling; the appropriate
product behaviour is to rank photos and move rejects to a review folder.

**7. A score-margin cut-off is no better than fixed top-k.** Keeping every photo
within δ of the group's best score performs equivalently at matched discard rates
(0.750 vs 0.777 at ~55% kept; 0.951 vs 0.933 at ~82% kept). The margin is a
monotone transform of the model's predicted win probability and therefore carries
no additional confidence information, so fixed top-k is retained for simplicity.

**Limitations.** Face-subset results rest on 255 pairs, where a handful of flipped
predictions changes significance, and several tests were run on that subset; they
are treated as hypothesis-generating. Photo Triage series average 2.7 photos,
shorter than real camera bursts, so both top-1 accuracy and the retention curve are
re-measured on a separate, people-heavy set of longer bursts collected and labelled
by the authors.

Standardising every feature by its training standard deviation before fitting changes test
accuracy by at most 0.4 points (CLIP only 0.641 to 0.637; CLIP plus hand features 0.652 to
0.655), so the reported results use unstandardised features.